# Streams

Welcome to the documentation the *Streams* part of *Kafi Streams*.

Streams offers easy-as-cake complex stateful stream processing based on pydbsp in the spirit of Kafka Streams.

## Topology

In Streams, you first create a *processing topology* or just *topology* with an arbitrary number of *sources* (=inputs) and an arbitrary number of *sinks* (=outputs).

A topology is written using a fluent API inspired by Kafka Streams (which, in turn, has been inspired by Java 8+ Streams).

The underlying class is called *TopologyNode* - which is basically just a thin wrapper around pydbsp.


### Example Topology

Let's proceed to an example topology.

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "join_1"
#
click_tn = (
    Tn.source(click_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"], "view_time": x["value"]["view_time"]})
    .distinct()
)
#
customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda x: {"id": x["value"]["id"], "name": x["value"]["name"]})
    .distinct()
)
#
join_1_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {"value": {
            "customer_id": l["customer_id"],
            "view_time": l["view_time"],
            "name": r["name"]}})
    .sink(sink_str)
)
#
built_tn = Tn.build(join_1_tn)


So what happens here? Let's have a look at the topology in visual form:


In [ ]:
print(built_tn.mermaid())

```mermaid
graph TD
b903e26b-a9e1-4d76-a0d0-af9427019696[map_op] --> 3dea6074-0fce-4007-8efa-0c499ef66563[distinct_op]
8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op] --> 13f25140-11c7-4beb-9bca-17a30aa9a155[sink_join_1]
64bae356-f252-40f7-a054-5f0b7abc68fe[source_clicks] --> 8894044d-22ce-4975-9cbd-1bb868660f96[map_op]
8894044d-22ce-4975-9cbd-1bb868660f96[map_op] --> bb683655-62c4-4950-8c44-9742ec74c431[distinct_op]
d775e1ce-ae9e-4640-892f-7faf148600a2[source_customers] --> b903e26b-a9e1-4d76-a0d0-af9427019696[map_op]
3dea6074-0fce-4007-8efa-0c499ef66563[distinct_op] --> 8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op]
bb683655-62c4-4950-8c44-9742ec74c431[distinct_op] --> 8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op]
```

We have two sources...
* `clicks`
* `customers`

...and one sink:
* `join_1`

We assume that `clicks` is a source of Kafka messages like this...

In [ ]:
click_message_dict = {
    "key": None,
    "value": {"customer_id": "4711",
              "view_time": 100},
    "partition": 2,
    "offset": 23,
    "timestamp": 1609457201000,
    "headers": None
}

...and `customers` is a source of Kafka messages like this:

In [ ]:
customer_message_dict = {
    "key": "4711",
    "value": {"id": "4711",
              "name": "Sallyann Jupp"},
    "partition": 0,
    "offset": 67,
    "timestamp": 1609457001000,
    "headers": None
}

The aim of the topology is to join `clicks` with `customers` to associate the clickstream IPs with the respective customer. An example output message looks like this (only the `value` field of the Kafka message is irrelevant here):

```python
{"value": {"customer_id": "4711",
           "view_time": 100,
           "name": "Sallyann Jupp"}
}
```

### How Does It Work?

How does the topology achieve its aim? It first selects the `customer_id` and `ip` fields from the `clicks` (`map`) and makes sure that the output of the selection contains no duplicates (`distinct`):

```python
click_tn = (
    Tn.source(click_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"], "view_time": x["value"]["view_time"]})
    .distinct()
)
```

Similarly, it selects `id` and `name` from the `customers` (`map` + `distinct`):
```python
customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda x: {"id": x["value"]["id"], "name": x["value"]["name"]})
    .distinct()
)
```

Next, the topology does an equi join (`join_equi`) on `customer_id` from the `clicks` and `id` from the `customers`:
```python
join_1_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {"value": {
            "customer_id": l["customer_id"],
            "view_time": l["view_time"],
            "name": r["name"]}})
    .sink(sink_str)
)
```
> You can use *any* field from the Kafka message for anything, including the equi join. We are not even restricted to `key` and `value`.

The final step "builds" the `TopologyNode` object by collecting its sinks:
```python
built_tn = Tn.build(join_1_tn)
```

That's it :-)


### Let's Run It!

Now let's see if it works...

In [ ]:
built_tn.push(click_source_str, [click_message_dict])
built_tn.push(customer_source_str, [customer_message_dict])
built_tn.latest()


What happens if we push the same messages once again?

In [ ]:
built_tn.push(click_source_str, [click_message_dict])
built_tn.push(customer_source_str, [customer_message_dict])
built_tn.latest()

Nothing happens! Why?

Because nothing has changed! We did not add any further information, just the same information again. The `distinct()` operators did the magic for us.

In classical stream processing, we needed to implement the concept of *exactly-once semantics*.

> In Kafi Streams, you can forget about exactly-once semantics. It boils down to simply using the right operator in the right place.


### But This Cannot Possibly Scale, Or Can It?

As you might have guessed, the `distinct()` operator (and also the `join_equi` operator, BTW) are stateful.

And yes - if we would confront this topology with an ever growing number of Kafka messages, we would soon run out of memory (and Kafi Streams would begin to crawl).

Let's see this happening.

We begin by writing generators for clicks and customers. For this, we need the "faker" library:

In [ ]:
!pip install faker

In [ ]:
import random, time

from faker import Faker

customers_int = 100

class ClickGenerator:
    def __init__(self):
        self.ts_int = int(time.time() * 1000)
        self.ts_step_int = 100
        self.customer_id_int = 0

    def generate(self):
        message_dict = {
            "key": None,
            "value": {"customer_id": random.randint(0, customers_int - 1),
                      "view_time": random.randint(10, 120),
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

class CustomerGenerator:
    def __init__(self):
        self.customer_id_int = 0
        self.customer_id_int_name_str_dict = {}
        fake = Faker()
        for customer_id_int in range(customers_int):
            name_str = fake.name()
            self.customer_id_int_name_str_dict[customer_id_int] = name_str

    def generate(self):
        customer_id_int = random.randint(0, customers_int - 1)
        message_dict = {
            "key": str(customer_id_int),
            "value": {"id": customer_id_int,
                      "name": self.customer_id_int_name_str_dict[customer_id_int]}
        }
        #
        return message_dict
    
click_generator = ClickGenerator()
for _ in range(3):
    print(click_generator.generate())

customer_generator = CustomerGenerator()
for _ in range(3):
    print(customer_generator.generate())


{'key': None, 'value': {'customer_id': 0, 'view_time': 103, 'ts': 1782630562846}}
{'key': None, 'value': {'customer_id': 0, 'view_time': 85, 'ts': 1782630562946}}
{'key': None, 'value': {'customer_id': 0, 'view_time': 52, 'ts': 1782630563046}}
{'key': '0', 'value': {'id': 0, 'name': 'Phillip Clayton'}}
{'key': '1', 'value': {'id': 1, 'name': 'Jennifer Russell'}}
{'key': '1', 'value': {'id': 1, 'name': 'Jennifer Russell'}}


Ok. Now we push 1000 * 100 clicks and 100 * 10 customers and measure the topology state size in KB after each step:

In [ ]:
import cloudpickle

built_tn.reset()

for i in range(1000):
    print(f"Step {i + 1}")
    click_message_dict_list = [click_generator.generate() for _ in range(100)]
    customer_message_dict_list = [customer_generator.generate() for _ in range(10)]
    built_tn.push(click_source_str, click_message_dict_list)
    built_tn.push(customer_source_str, customer_message_dict_list)
    print(built_tn.latest())
    print(len(cloudpickle.dumps(built_tn)) / 1024)


So... as expected - the topology state size increases and increases.

This truly doesn't scale. Yet.

It would if we found a way of letting older clicks *expire* such that pydbsp's garbage collector can purge them.

To this end, we need to augment our topology just a bit:


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "join_1"
#
click_tn = (
    Tn.source(click_source_str)
    ### Add expiration (after 10 click ts steps)
    .expire(lambda x: x["value"]["ts"],
            lambda x: x["value"]["ts"] + click_generator.ts_step_int * 10)
            # lambda x: (x["value"]["ts"] // (click_generator.ts_step_int * 10)) *  (click_generator.ts_step_int * 10) + (click_generator.ts_step_int * 10))
    ###
    .map(lambda x: {"customer_id": x["value"]["customer_id"], "view_time": x["value"]["view_time"]})
    .distinct()
)
#
customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda x: {"id": x["value"]["id"], "name": x["value"]["name"]})
    .distinct()
)
#
join_1_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {"value": {
            "customer_id": l["customer_id"],
            "view_time": l["view_time"],
            "name": r["name"]}})
    .sink(sink_str)
)
#
built_tn = Tn.build(join_1_tn)


We have added the `expire()` operator. Its arguments are:
1. A function for getting the timestamp of a message (here: `lambda x: x["value"]["ts"])`).
2. A function for getting the expiration time of this message (here: `lambda x: x["value"]["ts"])` = after 10 steps in the click generator)

Now we are ready to test again:


In [ ]:
built_tn.reset()

for i in range(1000):
    print(f"Step {i + 1}")
    click_message_dict_list = [click_generator.generate() for _ in range(100)]
    customer_message_dict_list = [customer_generator.generate() for _ in range(10)]
    built_tn.push(click_source_str, click_message_dict_list)
    built_tn.push(customer_source_str, customer_message_dict_list)
    print(built_tn.latest())
    print(len(cloudpickle.dumps(built_tn)) / 1024)


So... the state memory stays flat.

This does scale :-)

### How Does expire() Work?

In classical stream processing terms, the `expire()` operator implements a *per-event sliding window*. Under the covers:
1. Each individual message *m1* with timestamp *t_m1* is tagged with an expiration time *t_m1_e*.
2. Once one of the next messages *m2* has a timestamp *t_m2* > *t_m1_e*, message *m1* has *expired* and is marked for deletion.
3. In the next processing step, message *m1* is purged from the topology state by pydbsp's garbage collector.

Next, let's have a look at this in slow motion. Let's first add some debug output to our topology:

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "join_1"
#
click_tn = (
    Tn.source(click_source_str)
    .expire(lambda x: x["value"]["ts"],
    ### Set expiration to ts + 10
            lambda x: x["value"]["ts"] + 10,
    ###
    ### Add debug output
            _peek_input=True,
            _peek_current_time=True,
            _peek_expired=True)
    ###
    .map(lambda x: {"customer_id": x["value"]["customer_id"], "view_time": x["value"]["view_time"]})
    .distinct()
)
#
customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda x: {"id": x["value"]["id"], "name": x["value"]["name"]})
    .distinct()
)
#
join_1_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {"value": {
            "customer_id": l["customer_id"],
            "view_time": l["view_time"],
            "name": r["name"]}})._peek("output")
    .sink(sink_str)
)
#
built_tn = Tn.build(join_1_tn)

In [ ]:
built_tn.reset()

print("Step 1")
click_message_dict1 = {"value": {"customer_id": 1,
                                "view_time": 10,
                                "ts": 0}}
customer_message_dict1 = {"value": {"id": 1,
                                    "name": "Mary Shelley"}}
built_tn.push(click_source_str, [click_message_dict1])
built_tn.push(customer_source_str, [customer_message_dict1])
built_tn.latest()
print("")

print("Step 2")
click_message_dict2 = {"value": {"customer_id": 2,
                                "view_time": 20,
                                "ts": 12}}
customer_message_dict2 = {"value": {"id": 2,
                                    "name": "Douglas Adams"}}
built_tn.push(click_source_str, [click_message_dict2])
built_tn.push(customer_source_str, [customer_message_dict2])
built_tn.latest()
print("")

print("Step 3")
click_message_dict3 = {"value": {"customer_id": 1,
                                "view_time": 30,
                                "ts": 1}}
built_tn.push(click_source_str, [click_message_dict3])
built_tn.latest()
print("")

print("Step 4")
click_message_dict3 = {"value": {"customer_id": 1,
                                "view_time": 40,
                                "ts": 5}}
built_tn.push(click_source_str, [click_message_dict3])
built_tn.latest()
print("")


What happened here? We focus on the clicks:
* Step 1: Customer 1 (Mary Shelley) clicks at time (`ts`) `0`. The `expire()` operator tracks the current timestamp (now `0`). Since we set the expiry time to `ts + 10 = 10`, the click does not expire, goes into our join and the result is emitted to the output.
* Step 2: Customer 2 (Douglas Adams) clicks at time 12. The current timestamp is now `12`. The click by Mary Shelley above expires.
* Step 3: Mary Shelley clicks again at time `1`. The current timestamp stays unchanged at `12`. Mary Shelley's click comes too late. It expires immediately and is not emitted to the output.
* Step 4: Mary Shelley clicks once again at time `5`. The current timestamp once again stays unchanged at `12`. As the expiry time is `ts + 10 = 15`, this click goes through and is emitted to the output.


### Time Windows

In classical stream processing, a time window serves two purposes:
* *Garbage collection* - keep down RAM and disk (state stores) by garbage collecting expired messages
* *Emission control* -  keep down the number of emitted messages by grouping and aggregating them into time buckets

With the `expire()` operator, we only solve the first part (garbage collection).

So the next question is, how can we solve the second part (emission control)?

To this end, let's look at the four canonical time window types in classical stream processing:
1. Sliding
2. Tumbling
3. Hopping
4. Session

#### Sliding Windows

The `expire()` operator already gives us the behavior of a sliding window.

It does not give us emission control. All messages still go through downstream - except those coming in too late (they are discarded right away).

#### Tumbling Windows

Let's move on to tumbling windows. With only the `expire()` operator, what we are missing is emission control: Many use cases require grouping and aggregating the messages into time buckets in order to keep down the downstream traffic.

In classical stream processing and even in Feldera (based on the same mathematical underpinnings as Kafi Streams, i.e., DBSP), tumbling windows are implemented *outside* the actual stream processing logic. In Kafi Streams, we can easily implement them *inside* it.

Here is an example.

In [385]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "join_1"
#
tumbling_size_int = click_generator.ts_step_int * 10
#
click_tn = (
    Tn.source(click_source_str)
    ### Add expiration (after 10 click ts steps)
    .expire(lambda x: x["value"]["ts"],
            # lambda x: x["value"]["ts"] + click_generator.ts_step_int * 10)
            lambda x: (x["value"]["ts"] // tumbling_size_int) *  tumbling_size_int + tumbling_size_int * 10)
    ###
    .map(lambda x: {"customer_id": x["value"]["customer_id"], "view_time": x["value"]["view_time"], "ts": x["value"]["ts"]})
    .distinct()
)
#
customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda x: {"id": x["value"]["id"], "name": x["value"]["name"]})
    .distinct()
)
#
join_1_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {
            "customer_id": l["customer_id"],
            "view_time": l["view_time"],
            "name": r["name"],
            "ts": l["ts"]})
)
#
group_by_tn = (
    join_1_tn
    .group_by_agg(
        by_function=lambda x: (
            (x["ts"] // tumbling_size_int) * tumbling_size_int + tumbling_size_int,
            x["customer_id"]
        ),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_end": by[0],
            "customer_id": by[1],
            "name": agg["name"],
            "click_count": agg["click_count"],
            "total_view_time": agg["total_view_time"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "name": x["name"],
            "click_count": agg["click_count"] + w,
            "total_view_time": agg["total_view_time"] + x["view_time"] * w,
            "last_ts": max(agg["last_ts"], x["ts"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={
            "name": None,
            "click_count": 0,
            "total_view_time": 0,
            "last_ts": 0
        }
    )
)
#
current_window_tn = (
    join_1_tn
    .map(lambda x: x["ts"])
    .max(lambda x: x,
         lambda x: {"window_end": (x // tumbling_size_int) * tumbling_size_int + tumbling_size_int})
)

gatekeeper_tn = (
    group_by_tn
    .join(
        current_window_tn,
        lambda l, r: r["window_end"] > l["window_end"],
        lambda l, _: l
    )
)
# )._peek("gatekeeper")
#
# built_tn = Tn.build(join_1_tn)
# built_tn = Tn.build(group_by_tn)
built_tn = Tn.build(gatekeeper_tn)



In [380]:
import random, time

from faker import Faker

customers_int = 2

class ClickGenerator:
    def __init__(self):
        # self.ts_int = int(time.time() * 1000)
        self.ts_int = 0
        # self.ts_step_int = 100
        self.ts_step_int = 1
        self.customer_id_int = 0

    def generate(self):
        message_dict = {
            "key": None,
            "value": {"customer_id": random.randint(0, customers_int - 1),
                      "view_time": random.randint(10, 120),
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

class CustomerGenerator:
    def __init__(self):
        self.customer_id_int = 0
        self.customer_id_int_name_str_dict = {}
        fake = Faker()
        for customer_id_int in range(customers_int):
            name_str = fake.name()
            self.customer_id_int_name_str_dict[customer_id_int] = name_str

    def generate(self):
        customer_id_int = random.randint(0, customers_int - 1)
        message_dict = {
            "key": str(customer_id_int),
            "value": {"id": customer_id_int,
                      "name": self.customer_id_int_name_str_dict[customer_id_int]}
        }
        #
        return message_dict


In [ ]:
built_tn.reset()
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()


for i in range(100):
    print(f"Step {i + 1}")
    click_message_dict_list = [click_generator.generate() for _ in range(100)]
    customer_message_dict_list = [customer_generator.generate() for _ in range(10)]
    built_tn.push(click_source_str, click_message_dict_list)
    built_tn.latest()
    built_tn.push(customer_source_str, customer_message_dict_list)
    built_tn.latest()
    print(len(cloudpickle.dumps(built_tn)) / 1024)


Step 1
91.82421875
Step 2
205.373046875
Step 3
275.9775390625
Step 4
313.791015625
Step 5
329.6884765625
Step 6
331.0927734375
Step 7
331.6806640625
Step 8
331.841796875
Step 9
331.8271484375
Step 10
331.818359375
Step 11
331.8359375
Step 12
331.8388671875
Step 13
331.8447265625
Step 14
331.8505859375
Step 15
331.8447265625
Step 16
331.8447265625
Step 17
331.841796875
Step 18
331.8330078125
Step 19
331.8359375
Step 20
331.8447265625
Step 21
331.841796875
Step 22
331.8388671875
Step 23
331.84765625
Step 24
331.853515625
Step 25
331.841796875
Step 26
331.8359375
Step 27
331.8330078125
Step 28
331.818359375
Step 29
331.8271484375
Step 30
331.8388671875
Step 31
331.8271484375
Step 32
331.8359375
Step 33
331.8359375
Step 34
331.82421875
Step 35
331.8271484375
Step 36
331.8212890625
Step 37
331.82421875
Step 38
331.8447265625
Step 39
331.8359375
Step 40
331.8271484375
Step 41
331.8359375
Step 42
331.830078125
Step 43
331.830078125
Step 44
331.841796875
Step 45
331.84765625
Step 46
331.838867

KeyboardInterrupt: 